In [1]:
import os

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
  from google.colab import drive

  drive.mount('/content/drive') # load google drive
  os.chdir('/content/drive/My Drive/Thesis_Repository/Final_Google_Drive') # change directory to the current working directory

## Contingency Table 

In [2]:
import json
import pandas as pd

DATA_PATH = "train_test_split/train_stepverify_labeled_0.9.json"


# --------------------------------------------------
# 1. Load JSON file
# --------------------------------------------------
def load_data(path):
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    return data

# --------------------------------------------------
# 2. Extract pedagogy and mistake labels of the first Teacher
# --------------------------------------------------
def extract_pedagogy_and_mistake(data):
    rows = []
    missing_rows = []

    for item_index, item in enumerate(data):
        dialog_history = item.get("dialog_history", [])
        mistake_label = item.get("error_category")

        first_teacher_pedagogy = None
        first_teacher_text = None

        # Check only the actual first turn if the conversation is not empty
        if dialog_history:
            first_turn = dialog_history[0]

            if first_turn.get("user") == "teacher":
                # print(first_turn)
                first_teacher_pedagogy = first_turn.get("pedagogy")
                first_teacher_text = first_turn.get("text", "")

        # Exclude if pedagogy or mistake label for the first turn is missing
        if first_teacher_pedagogy is None or mistake_label is None:
            missing_rows.append({
                "item_index": item_index,
                "pedagogy": first_teacher_pedagogy,
                "mistake": mistake_label,
            })
            continue

        rows.append({
            "item_index": item_index,
            "pedagogy": first_teacher_pedagogy,
            "mistake": mistake_label,
            "teacher_text": first_teacher_text,
        })


    label_df = pd.DataFrame(rows)
    return label_df, missing_rows

def get_contingency_table(label_df):
    contingency_table = pd.crosstab(
        index=label_df["pedagogy"],
        columns=label_df["mistake"],
        margins=True,
        margins_name="Total",
    )

    return contingency_table

In [3]:
stepverify_data = load_data(DATA_PATH)
stepverify_label_df, stepverify_missing_rows = extract_pedagogy_and_mistake(stepverify_data)
print("Stepverify Data")
print(f"Total number of data points: {len(stepverify_data)}")
print(f"Number of data points used for analysis: {len(stepverify_label_df)}")
print(f"Number of excluded data points: {len(stepverify_missing_rows)}")

Stepverify Data
Total number of data points: 693
Number of data points used for analysis: 693
Number of excluded data points: 0


In [4]:
stepverify_contingency_table = get_contingency_table(stepverify_label_df)
stepverify_contingency_table

mistake,calculation_error_easily_solved_by_a_calculator,extra_quantity_or_missing_quantity,missing_wrong_factual_knowledge,misunderstanding_of_a_question,none_of_the_above,reached_correct_solution_but_proceeded_further,unit_conversion_error,Total
pedagogy,,,,,,,,
focus,11,10,8,9,5,4,0,47
generic,63,132,64,160,47,36,27,529
probing,12,22,21,30,8,9,6,108
telling,1,3,4,0,0,0,1,9
Total,87,167,97,199,60,49,34,693


In [12]:
import os

os.makedirs('pmi', exist_ok=True)
stepverify_contingency_table.to_csv('data_pmi_table/contingency_table.csv')
stepverify_label_df[['pedagogy', 'mistake']].to_csv('data_pmi_table/pedagogy_mistake_pair.csv', index=False)

## PMI, PPMI, and NPMI

### PMI

PMI (Pointwise Mutual Information) measures how strongly two events co-occur compared with what would be expected if they were independent. In this analysis, the two events are a `pedagogy` label and an `error_category` label.

$$
\mathrm{PMI}(x,y) = \log_2\left(\frac{P(x,y)}{P(x)P(y)}\right)
$$

- $P(x,y)$ is the joint probability of pedagogy $x$ and error category $y$.
- $P(x)$ and $P(y)$ are their marginal probabilities.
- $\mathrm{PMI} > 0$: the pair occurs more often than expected under independence.
- $\mathrm{PMI} = 0$: the pair occurs approximately as often as expected under independence.
- $\mathrm{PMI} < 0$: the pair occurs less often than expected under independence.

PMI ranges from $-\infty$ to $+\infty$. However, it can assign very large values to rare pairs, which makes direct comparison difficult when frequencies differ substantially.

### PPMI

PPMI (Positive PMI) keeps only positive associations by replacing negative PMI values with zero.

$$
\mathrm{PPMI}(x,y) = \max\left(\mathrm{PMI}(x,y), 0\right)
$$

This analysis uses PPMI because the goal is to identify pedagogy-error-category pairs that are positively associated. Negative associations are not interpreted as a separate ranking signal; only associations stronger than the independence expectation are retained.

### NPMI

NPMI (Normalized PMI) normalizes PMI by the information content of the joint event. This reduces the dependence of the score on the raw PMI scale.

$$
\mathrm{NPMI}(x,y) = \frac{\mathrm{PMI}(x,y)}{-\log_2 P(x,y)}
$$

NPMI usually ranges from $-1$ to $1$. Values close to $1$ indicate strong association, values close to $0$ indicate near independence, and negative values indicate less co-occurrence than expected. Because NPMI can still be unstable for very rare pairs, scores should be interpreted together with the corresponding co-occurrence counts.

### Zero Counts and Epsilon Smoothing

If $P(x,y)=0$, the logarithm is undefined. A small value, $\epsilon=10^{-12}$, can be introduced to avoid taking the logarithm of zero:

$$
\mathrm{PMI}_{\epsilon}(x,y) = \log_2\left(\frac{P(x,y)+\epsilon}{P(x)P(y)}\right)
$$

Adding a small epsilon to the joint probability is different from additive smoothing that adds a constant to every count in the contingency table. Röder et al. (2015) describe epsilon smoothing for avoiding the logarithm of zero, and report using $\epsilon=10^{-12}$ following Stevens et al. (2012). The very small value limits the effect of smoothing on observed associations while making zero-count cases computable.

### References

- Röder, M., Both, A., & Hinneburg, A. (2015). [Exploring the Space of Topic Coherence Measures](https://papers.dice-research.org/2015/WSDM_Palmetto/WSDM_palmetto_public.pdf)
- Stevens, K., Klingenstein, S., & others. (2012). [Exploring Topic Coherence over Many Models and Many Topics](https://aclanthology.org/D12-1087.pdf)
- [PMI, PPMI, and NPMI overview](https://www.emergentmind.com/topics/positive-pmi-ppmi)

In [6]:
import numpy as np
import pandas as pd

table = stepverify_contingency_table.drop(
    index="Total",
    columns="Total"
)

# joint probability
p_xy = table / table.values.sum()

# marginal probabilities
p_x = p_xy.sum(axis=1)   # row
p_y = p_xy.sum(axis=0)   # column

# epsilon smoothing 
epsilon = 1e-12

# PMI
pmi = np.log2((p_xy + epsilon) / np.outer(p_x, p_y))

pmi = pd.DataFrame(
    pmi,
    index=table.index,
    columns=table.columns
)

npmi = pmi / (-np.log2(p_xy) + epsilon)
ppmi = pmi.clip(lower=0) 

# 마지막에 출력용으로만 반올림
pmi = pmi
npmi = npmi
ppmi = ppmi

/Users/jay/Library/Caches/pypoetry/virtualenvs/thesis-env-8BIblUlj-py3.14/lib/python3.14/site-packages/pandas/core/internals/blocks.py:347: RuntimeWarning: divide by zero encountered in log2
  result = func(self.values, **kwargs)


In [8]:
npmi 

mistake,calculation_error_easily_solved_by_a_calculator,extra_quantity_or_missing_quantity,missing_wrong_factual_knowledge,misunderstanding_of_a_question,none_of_the_above,reached_correct_solution_but_proceeded_further,unit_conversion_error
pedagogy,,,,,,,
focus,0.150338,-0.029380,0.043844,-0.093282,0.041767,0.035958,-0.000000
generic,-0.021991,0.021016,-0.061200,0.035413,0.009605,-0.012936,0.012177
probing,-0.030103,-0.048711,0.094011,-0.010577,-0.034966,0.037825,0.026172
telling,-0.018667,0.059610,0.224141,-0.000000,-0.000000,-0.000000,0.124972


In [9]:
ppmi

mistake,calculation_error_easily_solved_by_a_calculator,extra_quantity_or_missing_quantity,missing_wrong_factual_knowledge,misunderstanding_of_a_question,none_of_the_above,reached_correct_solution_but_proceeded_further,unit_conversion_error
pedagogy,,,,,,,
focus,0.898611,0.000000,0.282210,0.000000,0.297160,0.267413,0.000000
generic,0.000000,0.050277,0.000000,0.074891,0.037286,0.000000,0.057012
probing,0.000000,0.000000,0.474229,0.000000,0.000000,0.237039,0.179324
telling,0.000000,0.468045,1.666874,0.000000,0.000000,0.000000,1.179324


In [10]:
stepverify_contingency_table

mistake,calculation_error_easily_solved_by_a_calculator,extra_quantity_or_missing_quantity,missing_wrong_factual_knowledge,misunderstanding_of_a_question,none_of_the_above,reached_correct_solution_but_proceeded_further,unit_conversion_error,Total
pedagogy,,,,,,,,
focus,11,10,8,9,5,4,0,47
generic,63,132,64,160,47,36,27,529
probing,12,22,21,30,8,9,6,108
telling,1,3,4,0,0,0,1,9
Total,87,167,97,199,60,49,34,693


In [13]:
pmi.to_csv("data_pmi_table/pmi_table.csv", index=True)
npmi.to_csv("data_pmi_table/npmi_table.csv", index=True)
ppmi.to_csv("data_pmi_table/ppmi_table.csv", index=True)